# 02 — Preprocessing

**Début** : charge `01_feature_engineering` (+ chemin images depuis `00`). **Fin** : manifest + state.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "shared").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

from shared.paths import CARDS_DIR, RAW_CARDS_DIR
from shared.pokemon_dataset import (
    build_tokenizer_and_loaders,
    collect_valid_image_paths,
    pick_device,
    save_preprocessing_manifest,
)
from shared.step_artifacts import (
    load_previous_step_output,
    load_step_output,
    rel_path,
    resolve_path,
    save_step_output,
    step_dir,
)

prev = load_previous_step_output("02_preprocessing")
pull = load_step_output("00_pull_data")

image_dir = resolve_path(pull["cards_dir"])
if not image_dir.exists() or not any(image_dir.rglob("*.png")):
    image_dir = CARDS_DIR if CARDS_DIR.exists() else RAW_CARDS_DIR

print(f"Images: {image_dir}")
print(f"Metadata exemple (étape 01): {prev.get('metadata_path')}")
device = pick_device()


[01_feature_engineering] État chargé ← /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/01_feature_engineering/state.json
[00_pull_data] État chargé ← /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/00_pull_data/state.json
Images: /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards
Metadata exemple (étape 01): artifacts/pipeline/steps/01_feature_engineering/example_metadata.json


In [2]:
IMG_SIZE = 512
BATCH_SIZE = 4
VAL_FRACTION = 0.1
SPLIT_SEED = 42

print(f"Using device: {device}")
image_paths = collect_valid_image_paths(image_dir)
print(f"Nombre d'images valides trouvées : {len(image_paths)}")
train_dataset, val_dataset, train_loader, val_loader, tokenizer = build_tokenizer_and_loaders(
    image_dir=image_dir,
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    val_fraction=VAL_FRACTION,
    split_seed=SPLIT_SEED,
)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

manifest_path = step_dir("02_preprocessing") / "preprocessing_manifest.json"
save_preprocessing_manifest(
    manifest_path,
    image_dir=image_dir,
    image_paths=image_paths,
    train_size=len(train_dataset),
    val_size=len(val_dataset),
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
)
print(f"Preprocessing manifest saved to: {manifest_path}")

save_step_output("02_preprocessing", {
    "cards_dir": pull["cards_dir"],
    "metadata_path": prev.get("metadata_path"),
    "manifest_path": rel_path(manifest_path),
    "n_images": len(image_paths),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
})
print("Step output saved for 02_preprocessing")

Using device: cpu
Found 15605 image file(s) under /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards, validating metadata and integrity...
15605 valid image(s) with pokemon_metadata found under /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards
Nombre d'images valides trouvées : 15605
Found 15605 image file(s) under /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards, validating metadata and integrity...
15605 valid image(s) with pokemon_metadata found under /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards
Found 15605 valid image(s) with pokemon_metadata under /home/light/Documents/DO5/MLOps/mlops-pokegen/data/raw/cards
Train: 14045, Val: 1560
Train dataset size: 14045
Validation dataset size: 1560
Manifest enregistré: /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/02_preprocessing/preprocessing_manifest.json
Preprocessing manifest saved to: /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/02_preproc